#  **SETUP**

In [0]:
from pyspark.sql.functions import (col, count, avg, sum, min, max, countDistinct, when, length, trim, initcap,length, lpad)

In [0]:
## Load Tables Dataframe

customers_df = spark.table(
    "workspace.shop_flow_bronze.customers"
)

orders_df = spark.table(
    "workspace.shop_flow_bronze.orders"
)

order_items_df = spark.table(
    "workspace.shop_flow_bronze.order_items"
)

products_df = spark.table(
    "workspace.shop_flow_bronze.products"
)

sellers_df = spark.table(
    "workspace.shop_flow_bronze.sellers"
)

order_payments_df = spark.table(
    "workspace.shop_flow_bronze.order_payments"
)

order_reviews_df = spark.table(
    "workspace.shop_flow_bronze.order_reviews"
)

product_categories_df = spark.table(
    "workspace.shop_flow_bronze.product_category_translation"
)

geolocation_df = spark.table(
    "workspace.shop_flow_bronze.geolocation"
)

#  **TABLES profile**

#  **CUSTOMERS** (Profiling/Validation)

In [0]:
customers_df.display()

## Duplication/Null Key

In [0]:
## Check the key if Null

customers_df.filter(
    col("customer_id").isNull()
    ).display()

In [0]:
## Check the duplicates

customers_df.groupBy("customer_id").count().filter(col("count")> 1).display()

In [0]:
## Check the unique_id if Null

customers_df.filter(
    col("customer_unique_id").isNull()
    ).count()

In [0]:
## check empty string

customers_df.select(
    sum(when(col("customer_id") == "", 1).otherwise(0)).alias("empty_customer_id"),
    sum(when(col("customer_unique_id") == "", 1).otherwise(0)).alias("empty_customer_unique_id"),
    sum(when(col("customer_city") == "", 1).otherwise(0)).alias("empty_city"),
    sum(when(col("customer_state") == "", 1).otherwise(0)).alias("empty_state")
).display()

In [0]:
## Check ZIP codes if different than 5 digits



customers_df.select("customer_zip_code_prefix") \
    .filter(
        length(col("customer_zip_code_prefix").cast("string")) != 5 
    ) \
    .count()

## Check Standarization for city

In [0]:
customers_df.select("customer_city").distinct().display()

# GEOLOCATION

In [0]:
geolocation_df.display()

# ORDER ITEMS

In [0]:
order_items_df.display()

## Duplication/Nulls

In [0]:
# Check the duplication of the order_id by one order_item_id

order_items_df.groupBy("order_id","order_item_id").count().filter(col("count")> 1).display()

In [0]:
# Check the primary keys if null

order_items_df.filter(col("order_item_id").isNull()).display()

order_items_df.filter(col("order_id").isNull()).display()

order_items_df.filter(col("product_id").isNull()).display()

order_items_df.filter(col("seller_id").isNull()).display()

## Invalid Numeric Values

In [0]:
# Check the validation of price, freight_value numeric values

order_items_df.filter(col("price") < 0).display()
order_items_df.filter(col("freight_value") < 0).display()


In [0]:
missing_products = order_items_df.join(
    products_df.select("product_id").distinct(),
    on="product_id",
    how="left_anti"
)

missing_products.display()

In [0]:
order_items_df.printSchema()

# ORDER_PAYMENTS

In [0]:
order_payments_df.display()

## Duplication/Nulls

In [0]:
# Check duplication order_id by payment_sequential

order_payments_df.groupBy("order_id","payment_sequential").count().filter(col("count")>1).display()



In [0]:
# Check values if null

order_payments_df.filter(col("order_id").isNull()).display()

order_payments_df.filter(col("payment_sequential").isNull()).display()

order_payments_df.filter(col("payment_type").isNull()).display()

order_payments_df.filter(col("payment_installments").isNull()).display()

order_payments_df.filter(col("payment_value").isNull()).display()



## Standarization

In [0]:
order_payments_df.select("payment_type").distinct().display()



## Invalid Numeric Values

In [0]:
# Check the validation of price, freight_value numeric values

order_payments_df.filter(col("payment_installments") < 0).display()
order_payments_df.filter(col("payment_value") < 0).display()

# ORDER_REVIEWS

In [0]:
order_revies_df.display()
# Check duplication review_id

## Duplication

In [0]:
order_reviews_df.groupBy("review_id") \
    .count() \
    .filter(col("count") > 1) \
    .display()

# 2. Check multiple review records per order
order_reviews_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .display()

# 3. Inspect the schema
order_reviews_df.printSchema()

# 4. Inspect review scores
order_reviews_df.groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .display()

In [0]:
# Check the duplicated review_id joining with the table order reviews Pyspark

from pyspark.sql.functions import col, count

duplicate_review_ids = (
    order_reviews_df
    .groupBy("review_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

order_reviews_df.join(
    duplicate_review_ids.select("review_id"),
    on="review_id",
    how="inner"
).orderBy("review_id", "order_id").display()

In [0]:
%sql

-- Check the duplicated review_id joining with the table order reviews Pyspark

WITH duplicate_review_ids AS(
    SELECT 
        review_id,
        COUNT(*) AS row_count
    FROM workspace.shop_flow_bronze.order_reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
)
SELECT o.*
FROM duplicate_review_ids d
INNER JOIN workspace.shop_flow_bronze.order_reviews o
ON d.review_id = o.review_id
ORDER BY d.review_id 

In [0]:
%sql
SELECT 
    *
FROM workspace.shop_flow_bronze.order_reviews or
INNER JOIN workspace.shop_flow_bronze.orders o
ON o.order_id = or.order_id
WHERE review_id = '00130cbe1f9d422698c812ed8ded1919'

In [0]:
## Check the if there are duplicated rows for combining ("review_id", "order_id")

from pyspark.sql.functions import col, count

duplicate_review_order_pairs = (
    order_reviews_df
    .groupBy("review_id", "order_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

duplicate_review_order_pairs.display()

In [0]:
# Check for null values

order_reviews_df.filter(col("review_id").isNull()).display()

order_reviews_df.filter(col("order_id").isNull()).display()



# ORDERS

In [0]:
orders_df.display()

orders_df.printSchema()

## Duplication

In [0]:
# check the duplication of order_id

orders_df.groupBy("order_id").count().filter(col("count") > 1).display()

# check the duplication of order_id

orders_df.groupBy("customer_id").count().filter(col("count") > 1).display()


## Primary key Nulls

In [0]:
# Check if order primary key is Null

orders_df.filter(col("order_id").isNull()).display()

# Check if customer_id key is Null

orders_df.filter(col("customer_id").isNull()).display()

## Standarization

In [0]:
orders_df.select("order_status").distinct().display()

# PRODUCT CATEGORY TRANSALATION

In [0]:
product_categories_df.display()

# PRODUCTS

In [0]:
products_df.display()


## Duplication/Nulls

In [0]:
# Check duplication for primary key

products_df.groupBy("product_id").count().filter(col("count") > 1).display()

# Check if product_id is Null

products_df.filter(col("product_id").isNull()).display()




## Unwanted Spaces

In [0]:
# Check unwanted spaces for primary key

products_df.filter(col("product_id") != trim(col("product_id"))).display()

# Check unwanted spaces for the category name

products_df.filter(col("product_category_name") != trim(col("product_category_name"))).display()

## Standarization

In [0]:
products_df.select("product_category_name").distinct().display()

In [0]:
from pyspark.sql.functions import col, sum, when 

products_df.select(
    sum(when(col("product_category_name").isNull(), 1)
        .otherwise(0)).alias("null_category"),
    sum(when(col("product_name_lenght").isNull(), 1)
        .otherwise(0)).alias("null_name_lengh"),
    sum(when(col("product_description_lenght").isNull(), 1)
        .otherwise(0)).alias("null_description_lenght"),
    sum(when(col("product_photos_qty").isNull(), 1)
        .otherwise(0)).alias("null_photos_qty"),
    sum(when(col("product_weight_g").isNull(), 1)
        .otherwise(0)).alias("null_weight_g"),
    sum(when(col("product_length_cm").isNull(), 1)
        .otherwise(0)).alias("null_length_cm"),
    sum(when(col("product_height_cm").isNull(), 1)
        .otherwise(0)).alias("null_height_cm"),
    sum(when(col("product_width_cm").isNull(), 1)
        .otherwise(0)).alias("null_width_cm"),
).display()

In [0]:
measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column_name in measurement_columns:
    products_df.filter(
        col(column_name).isNotNull() &
        (col(column_name) <= 0)
    ).select(
        "product_id", column_name
    ).display()

In [0]:
products_df.filter(
    col("product_weight_g") == 0
).display()

In [0]:
from pyspark.sql.functions import col

products_df.filter(
    (col("product_category_name").isNull() !=
     col("product_name_lenght").isNull()) |
    (col("product_category_name").isNull() !=
     col("product_description_lenght").isNull()) |
    (col("product_category_name").isNull() !=
     col("product_photos_qty").isNull())
).count()

# SELLERS

In [0]:
sellers_df.display()

## Duplication/Nulls

In [0]:
sellers_df.groupBy("seller_id").count().filter(col("count") > 1).display()

sellers_df.filter(col("seller_id").isNull()).display()

## Check invalid zip codes

In [0]:
sellers_df.printSchema()

In [0]:
sellers_df.filter(length(col("seller_zip_code_prefix")) != 5).display()

## Check unwanted spaces

In [0]:
sellers_df.filter(trim(col("seller_city")) != col("seller_city")).display()

sellers_df.filter(trim(col("seller_state")) != col("seller_state")).display()

## Standarization

In [0]:
sellers_df.select("seller_city").distinct().display()

sellers_df.select("seller_state").distinct().display()